Custom Model Implementation Usage

Christian Basso

CSC 5601

December 3rd, 2024

# Custom Model Implementation Usage

In [1]:
import pandas as pd
import numpy as np
from coxph_model import CoxPHModel
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
warnings.filterwarnings('ignore')


In [2]:
data = pd.read_csv('../data/clean_data.csv')
data = data.astype({col: 'int' for col in data.select_dtypes(include=['bool']).columns})

We need to split the data a little differently. I implimented the fit method to take in the time, event, and coariantes in different arrays.

In [3]:
train_df, test_df = train_test_split(data, test_size=0.2, random_state=42)

# Separate features and targets for fitting
X_train = train_df.drop(columns=['Overall Survival (Months)', 'Overall Survival Status'])
T_train = train_df['Overall Survival (Months)']
E_train = train_df['Overall Survival Status']

X_test = test_df.drop(columns=['Overall Survival (Months)', 'Overall Survival Status'])
T_test = test_df['Overall Survival (Months)']
E_test = test_df['Overall Survival Status']


X_train_values = X_train.values
T_train_values = T_train.values
E_train_values = E_train.values

Train the model

In [4]:
cox_model = CoxPHModel()

Benchmark fitting

In [5]:
%%time
cox_model.fit(X_train_values, T_train_values, E_train_values)

CPU times: total: 531 ms
Wall time: 2.36 s


RMSE

In [6]:
X_test_values = X_test.values
predictions = cox_model.predict_expected_survival_time(X_test_values)
test_df["PredictedMedianSurvivalTime"] = predictions

rmse = np.sqrt(mean_squared_error(test_df["Overall Survival (Months)"], test_df["PredictedMedianSurvivalTime"]))
print(f"RMSE: {rmse}")

RMSE: 97.82135481452593


Benchmark prediction on predict

In [7]:
%%time 
predictions = cox_model.predict_expected_survival_time(X_test_values)

CPU times: total: 0 ns
Wall time: 2.01 ms
